# Подготовка датасета: поликлиника

Таблица доступности строится по режиму работы: дата и время приёма и признак, можно ли записаться.
По её строкам собираются вопросы вида «можно ли записаться на такой день и время».
Справочные вопросы собираются по карточке учреждения и списку специалистов.

Справочные вопросы генерирует локальная модель Ollama `qwen3.8:27b`.
В `dataset.json` сохраняются пары `{kind, question, answer}`, таблица слотов пишется отдельно в `availability.json`.

In [1]:
import json
import random
import re
from collections import defaultdict
from datetime import date, datetime, time
from pathlib import Path

from tqdm import tqdm

from universal_consultant import generate_availability, generate_client_questions


def project_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError("Не найден корень проекта.")


DATA = project_root() / "data" / "polyclinics"
START = date(2026, 11, 16)
DAYS = 14
SLOT_MINUTES = 30
KNOWLEDGE_COUNT = 3
AVAILABILITY_LIMIT = 4
SEED = 7
knowledge: list[dict[str, str]] = []
APPOINTMENT_TEMPLATES = (
    "Можно ли записаться к специалисту {subject} на {when}?",
    "Есть ли запись к специалисту {subject} на {when}?",
    "Можно ли попасть на приём к специалисту {subject} {when}?",
)
_MONTHS = (
    "января",
    "февраля",
    "марта",
    "апреля",
    "мая",
    "июня",
    "июля",
    "августа",
    "сентября",
    "октября",
    "ноября",
    "декабря",
)
_WEEKDAYS = {"пн": 0, "вт": 1, "ср": 2, "чт": 3, "пт": 4, "сб": 5, "вс": 6}
_SCHEDULE_LINE = re.compile(
    r"(?P<days>[А-Яа-яЁё]{2}(?:\s*[-,]\s*[А-Яа-яЁё]{2})*)\s*:\s*"
    r"(?P<start>\d{1,2}:\d{2})\s*-\s*(?P<end>\d{1,2}:\d{2})",
    re.IGNORECASE,
)


def _format_moment(value: date | datetime) -> str:
    text = f"{value.day} {_MONTHS[value.month - 1]} {value.year} года"
    if isinstance(value, datetime):
        text += f" в {value:%H:%M}"
    return text


def _sample(items: list, limit: int | None, seed: int) -> list:
    if limit is None or limit >= len(items):
        return list(items)
    if limit < 1:
        raise ValueError("limit должен быть не меньше 1.")
    free = [item for item in items if item.available]
    busy = [item for item in items if not item.available]
    rng = random.Random(seed)
    rng.shuffle(free)
    rng.shuffle(busy)
    free_count = min(len(free), limit // 2)
    busy_count = min(len(busy), limit - free_count)
    if free_count + busy_count < limit and len(free) > free_count:
        free_count += min(limit - free_count - busy_count, len(free) - free_count)
    elif free_count + busy_count < limit and len(busy) > busy_count:
        busy_count += min(limit - free_count - busy_count, len(busy) - busy_count)
    picked = free[:free_count] + busy[:busy_count]
    rng.shuffle(picked)
    return picked


def appointment_questions(
    rows, templates, *, subject: str = "", limit: int | None = None, seed: int = 0
):
    """Вопрос про одно окно записи. Ответ — признак этой строки таблицы."""
    patterns = [template.strip() for template in templates if template.strip()]
    if not patterns:
        raise ValueError("Нужен хотя бы один шаблон вопроса.")
    name = subject.strip()
    questions = []
    for index, row in enumerate(_sample(list(rows), limit, seed)):
        questions.append(
            {
                "subject": name,
                "question": patterns[index % len(patterns)].format(
                    when=_format_moment(row.date),
                    subject=name,
                ),
                "date": row.date,
                "available": row.available,
            }
        )
    return questions


def parse_week_schedule(text: str) -> dict[int, tuple[time, time]]:
    """Разобрать режим работы вида «Пн-Пт: 08:00-20:00».

    Ключ — день недели, понедельник это 0. Дня нет в словаре — в этот день закрыто.
    """
    schedule: dict[int, tuple[time, time]] = {}
    for match in _SCHEDULE_LINE.finditer(text):
        start = _clock(match.group("start"))
        end = _clock(match.group("end"))
        if start >= end:
            raise ValueError("Интервал часов в расписании пустой.")
        for weekday in _expand_days(match.group("days")):
            schedule[weekday] = (start, end)
    if not schedule:
        raise ValueError("В тексте нет расписания по дням недели.")
    return schedule


def _clock(value: str) -> time:
    hour_text, minute_text = value.split(":")
    hour, minute = int(hour_text), int(minute_text)
    if hour > 23 or minute > 59:
        raise ValueError(f"Некорректное время в расписании: {value}.")
    return time(hour, minute)


def _expand_days(raw: str) -> list[int]:
    days: list[int] = []
    for part in re.split(r"\s*,\s*", raw.strip()):
        bounds = [item.strip().lower() for item in part.split("-")]
        if not bounds or any(item not in _WEEKDAYS for item in bounds):
            raise ValueError(f"Неизвестный день недели: {part}.")
        if len(bounds) == 1:
            days.append(_WEEKDAYS[bounds[0]])
            continue
        if len(bounds) != 2:
            raise ValueError(f"Неизвестный день недели: {part}.")
        start, end = _WEEKDAYS[bounds[0]], _WEEKDAYS[bounds[1]]
        if end < start:
            raise ValueError(f"Интервал дней недели пустой: {part}.")
        days.extend(range(start, end + 1))
    return days


def read_json(name: str):
    return json.loads((DATA / name).read_text(encoding="utf-8"))

In [3]:
info = read_json("info.json")
specialists = read_json("specialists.json")

by_role: dict[str, list[str]] = defaultdict(list)
for person in specialists:
    by_role[person["specialist"]].append(person["name"])
roles = sorted(by_role)

specialist_document = "\n".join(f"{role}: {', '.join(by_role[role])}" for role in roles)
documents = [{"name": item["name"], "fulltext": item["fulltext"]} for item in info]
documents.append({"name": "Специалисты", "fulltext": specialist_document})
len(documents), len(roles)

(5, 21)

In [5]:
knowledge = []
for document in tqdm(documents):
    questions = generate_client_questions(
        document["fulltext"],
        subject=None,
        count=KNOWLEDGE_COUNT,
    )
    knowledge.extend(
        {
            "kind": "knowledge",
            "source": document["name"],
            "question": item.question,
            "answer": item.answer,
            "document": document["fulltext"],
        }
        for item in questions
    )

len(knowledge)

100%|██████████| 5/5 [01:37<00:00, 19.55s/it]


15

In [7]:
hours = next(item["fulltext"] for item in info if item["name"] == "Режим работы")
open_hours = parse_week_schedule(hours)

availability = []
appointments = []

for index, role in enumerate(roles):
    slots = generate_availability(
        start=START,
        days=DAYS,
        with_time=True,
        slot_minutes=SLOT_MINUTES,
        open_hours=open_hours,
        seed=SEED + index,
    )
    availability.append(
        {
            "subject": role,
            "slots": [
                {"date": slot.date.isoformat(), "available": slot.available} for slot in slots
            ],
        }
    )
    appointments.extend(
        {
            "kind": "availability",
            "subject": item["subject"],
            "question": item["question"],
            "date": item["date"].isoformat(),
            "available": item["available"],
            "answer": "Да" if item["available"] else "Нет",
        }
        for item in appointment_questions(
            slots,
            APPOINTMENT_TEMPLATES,
            subject=role,
            limit=AVAILABILITY_LIMIT,
            seed=SEED + index,
        )
    )

len(availability), len(appointments)

(21, 84)

In [8]:
sample = availability[0]
print(sample["subject"])
print(f"{'date':<20} available")
for row in sample["slots"][:12]:
    print(f"{row['date']:<20} {row['available']}")
print()
for item in appointments[:6]:
    print(item["available"], item["question"])

Врач общей практики
date                 available
2026-11-16T08:00:00  True
2026-11-16T08:30:00  True
2026-11-16T09:00:00  False
2026-11-16T09:30:00  True
2026-11-16T10:00:00  False
2026-11-16T10:30:00  True
2026-11-16T11:00:00  True
2026-11-16T11:30:00  False
2026-11-16T12:00:00  True
2026-11-16T12:30:00  True
2026-11-16T13:00:00  True
2026-11-16T13:30:00  True

True Можно ли записаться к специалисту Врач общей практики на 17 ноября 2026 года в 12:30?
True Есть ли запись к специалисту Врач общей практики на 17 ноября 2026 года в 08:30?
False Можно ли попасть на приём к специалисту Врач общей практики 19 ноября 2026 года в 17:30?
False Можно ли записаться к специалисту Врач общей практики на 27 ноября 2026 года в 16:30?
True Можно ли записаться к специалисту Врач по лечебной физкультуре на 16 ноября 2026 года в 12:30?
False Есть ли запись к специалисту Врач по лечебной физкультуре на 28 ноября 2026 года в 16:00?


In [9]:
dataset = [
    {
        "kind": item["kind"],
        "question": item["question"],
        "answer": item["answer"],
    }
    for item in knowledge + appointments
]
(DATA / "dataset.json").write_text(
    json.dumps(dataset, ensure_ascii=False, indent=4) + "\n",
    encoding="utf-8",
)
(DATA / "availability.json").write_text(
    json.dumps(availability, ensure_ascii=False, indent=4) + "\n",
    encoding="utf-8",
)
len(dataset)

99